<a href="https://colab.research.google.com/github/Bilal-data/AuditGuard/blob/main/AuditGuard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AuditGuard

Financial Intelligence & Risk Analytics Platform

1. Project Objective

AuditGuard is a financial analytics platform designed to identify unusual transactions, potential fraud indicators, duplicate payments, and financial control weaknesses.

2. Business Problem

Organizations process large volumes of financial transactions. Manual auditing can be time-consuming and may fail to identify unusual patterns.

3. Proposed Solution

AuditGuard analyzes financial data using SQL, Python, statistical techniques, and anomaly detection methods to highlight transactions that require further investigation.

4. Core Modules

- Financial Data Quality
- Transaction Analysis
- Duplicate Payment Detection
- Suspicious Pattern Detection
- Risk Scoring
- Financial Risk Dashboard
- Audit Reporting

5. Target Users

- Financial Analysts
- Internal Auditors
- Risk Analysts
- Compliance Teams

6. Important Limitation

AuditGuard identifies risk indicators. It does not independently establish fraud, theft, or individual responsibility. All alerts require human investigation.

7. Technology Stack

- Python
- Pandas
- SQL / DuckDB
- Google Colab
- GitHub
- Power BI or a web-based dashboard

# AuditGuard
## Day 02 - Duplicate Invoice Detection

Objective:
Identify potential duplicate invoices using Python and Pandas.

In [ ]:
import pandas as pd

data = {
    "invoice_id": ["INV-101",
"INV-102", "INV-103",
                   "INV-104", "INV-105", "INV-106"],
                          "supplier": ["Supplier A", "Supplier B", "Supplier A",
                                            "Supplier C", "Supplier B", "Supplier A"],
                                                "amount": [1000, 500, 1000, 8000, 500, 1500],
                                                    "invoice_date": [
                                                            "2026-11-01", "2026-11-02", "2026-11-01",
                                                                    "2026-11-03", "2026-11-02", "2026-11-04"
                                                                        ]
}

df = pd.DataFrame(data)

df["invoice_date"] = pd.to_datetime(df["invoice_date"])

print(df)

  invoice_id    supplier  amount invoice_date
0    INV-101  Supplier A    1000   2026-11-01
1    INV-102  Supplier B     500   2026-11-02
2    INV-103  Supplier A    1000   2026-11-01
3    INV-104  Supplier C    8000   2026-11-03
4    INV-105  Supplier B     500   2026-11-02
5    INV-106  Supplier A    1500   2026-11-04


In [ ]:

# Find invoices with the same supplier and amount

df["potential_duplicate"] = df.duplicated(
    subset=["supplier", "amount"],
    keep=False)

duplicates = df[df["potential_duplicate"]]

print(duplicates)


  invoice_id    supplier  amount invoice_date  potential_duplicate
0    INV-101  Supplier A    1000   2026-11-01                 True
1    INV-102  Supplier B     500   2026-11-02                 True
2    INV-103  Supplier A    1000   2026-11-01                 True
4    INV-105  Supplier B     500   2026-11-02                 True


In [ ]:

# Improve duplicate detection using supplier,
# amount, and invoice date

df["duplicate_flag"] = df.duplicated(
    subset=["supplier", "amount", "invoice_date"],
        keep=False
        )

df["risk_status"] =df["duplicate_flag"].map({
            True: "REVIEW",
                False: "NORMAL"
                })

print(df[[
                    "invoice_id",
                        "supplier",
                            "amount",
                                "invoice_date",
                                    "risk_status"
                                    ]])


  invoice_id    supplier  amount invoice_date risk_status
0    INV-101  Supplier A    1000   2026-11-01      REVIEW
1    INV-102  Supplier B     500   2026-11-02      REVIEW
2    INV-103  Supplier A    1000   2026-11-01      REVIEW
3    INV-104  Supplier C    8000   2026-11-03      NORMAL
4    INV-105  Supplier B     500   2026-11-02      REVIEW
5    INV-106  Supplier A    1500   2026-11-04      NORMAL


In [ ]:

total_invoices = len(df)
flagged_invoices = df["duplicate_flag"].sum()
normal_invoices = total_invoices - flagged_invoices

print("AUDITGUARD - DAILY REPORT")
print("-------------------------")
print("Total invoices:", total_invoices)
print("Invoices flagged:", flagged_invoices)
print("Normal invoices:", normal_invoices)
print("Flagged percentage:",
      round(flagged_invoices /
total_invoices * 100, 2), "%")


AUDITGUARD - DAILY REPORT
-------------------------
Total invoices: 6
Invoices flagged: 4
Normal invoices: 2
Flagged percentage: 66.67 %



## Day 02 - Results

### Objective
Detect potential duplicate invoices.

### Method
Used Pandas duplicated() with supplier,
amount, and invoice_date.

### Results
- Total invoices: 6
- Flagged invoices: 4
- Normal invoices: 2

### Business Insight
Potential duplicate invoices should be reviewed
before payment approval.

### Limitation
Matching supplier, amount, and date does not
prove fraud or duplicate payment.
Human verification is required.


In [ ]:

!pip install -q duckdb


In [ ]:

import duckdb

# Create database connection
con = duckdb.connect()

# Register the existing Pandas DataFrame
con.register("invoices_df", df)

# Create invoices table
con.execute("""
    CREATE OR REPLACE TABLE invoices AS
        SELECT *
            FROM invoices_df
            """)

print("AuditGuard SQL database is ready!")


AuditGuard SQL database is ready!


In [ ]:

result = con.execute("""
    SELECT *
        FROM invoices
        """).df()

display(result)


,invoice_id,supplier,amount,invoice_date,potential_duplicate,duplicate_flag,risk_status
0,INV-101,Supplier A,1000,2026-11-01,True,True,REVIEW
1,INV-102,Supplier B,500,2026-11-02,True,True,REVIEW
2,INV-103,Supplier A,1000,2026-11-01,True,True,REVIEW
3,INV-104,Supplier C,8000,2026-11-03,False,False,NORMAL
4,INV-105,Supplier B,500,2026-11-02,True,True,REVIEW
5,INV-106,Supplier A,1500,2026-11-04,False,False,NORMAL


In [ ]:

duplicate_groups = con.execute("""
    SELECT
        supplier,
        amount,
        invoice_date,
        COUNT(*) AS invoice_count

    FROM invoices

           GROUP BY
              supplier,
              amount,
              invoice_date

    HAVING COUNT(*) > 1

     ORDER BY invoice_count DESC
""").df()

display(duplicate_groups)


,supplier,amount,invoice_date,invoice_count
0,Supplier B,500,2026-11-02,2
1,Supplier A,1000,2026-11-01,2


In [ ]:
suspicious_invoices = con.execute("""
    WITH duplicate_groups AS (
            SELECT
                supplier,
                amount,
                invoice_date

            FROM invoices

            GROUP BY
               supplier,
               amount,
               invoice_date

            HAVING COUNT(*) > 1
  )

  SELECT
    i.invoice_id,
    i.supplier,
    i.amount,
    i.invoice_date

  FROM invoices AS i

  INNER JOIN duplicate_groups AS d
     ON i.supplier = d.supplier
     AND i.amount = d.amount
     AND i.invoice_date =
d.invoice_date

    ORDER BY i.supplier
""").df()
display(suspicious_invoices)


,invoice_id,supplier,amount,invoice_date
0,INV-101,Supplier A,1000,2026-11-01
1,INV-103,Supplier A,1000,2026-11-01
2,INV-102,Supplier B,500,2026-11-02
3,INV-105,Supplier B,500,2026-11-02


In [ ]:

result = con.execute("""
    SELECT *
    FROM invoices
""").df()

display(result)


,invoice_id,supplier,amount,invoice_date,potential_duplicate,duplicate_flag,risk_status
0,INV-101,Supplier A,1000,2026-11-01,True,True,REVIEW
1,INV-102,Supplier B,500,2026-11-02,True,True,REVIEW
2,INV-103,Supplier A,1000,2026-11-01,True,True,REVIEW
3,INV-104,Supplier C,8000,2026-11-03,False,False,NORMAL
4,INV-105,Supplier B,500,2026-11-02,True,True,REVIEW
5,INV-106,Supplier A,1500,2026-11-04,False,False,NORMAL


In [ ]:

duplicate_groups = con.execute("""
    SELECT
        supplier,
          amount,
          invoice_date,
          COUNT(*) AS invoice_count

    FROM invoices

        GROUP BY
        supplier,
        amount,
        invoice_date

    HAVING COUNT(*) > 1

    ORDER BY invoice_count DESC
""").df()
display(duplicate_groups)


,supplier,amount,invoice_date,invoice_count
0,Supplier B,500,2026-11-02,2
1,Supplier A,1000,2026-11-01,2


In [57]:

suspicious_invoices = con.execute("""
    WITH duplicate_groups AS (
            SELECT
                supplier,
                amount,
                invoice_date
            FROM invoices

            GROUP BY
               supplier,
               amount,
               invoice_date
            HAVING COUNT(*) > 1
    )

            SELECT
                                                                                                                            i.invoice_id,
                                                                                                                                    i.supplier,
                                                                                                                                            i.amount,
                                                                                                                                                    i.invoice_date

                                                                                                                                                        FROM invoices AS i

                                                                                                                                                            INNER JOIN duplicate_groups AS d
                                                                                                                                                                    ON i.supplier = d.supplier
                                                                                                                                                                            AND i.amount = d.amount
                                                                                                                                       AND i.invoice_date = d.invoice_date

                                                                                                                                             ORDER BY
                                                                                                                                                    i.supplier,
                                                                                                                                                     i.invoice_date
                                                                                                                                                                  """).df()
display(suspicious_invoices)


,invoice_id,supplier,amount,invoice_date
0,INV-101,Supplier A,1000,2026-11-01
1,INV-103,Supplier A,1000,2026-11-01
2,INV-102,Supplier B,500,2026-11-02
3,INV-105,Supplier B,500,2026-11-02


In [58]:

print("AUDITGUARD - SQL AUDIT REPORT")
print("--------------------------------")

print("Total invoices:", len(df))

print(
    "Potential duplicate invoices:",
        len(suspicious_invoices)
        )

print(
      "Total flagged amount:",
      suspicious_invoices["amount"].sum()
    )


AUDITGUARD - SQL AUDIT REPORT
--------------------------------
Total invoices: 6
Potential duplicate invoices: 4
Total flagged amount: 3000



## Day 03 - Advanced SQL Analysis

### SQL Concepts
- SELECT
- GROUP BY
- HAVING
- CTE
- INNER JOIN

### Achievement
Extracted potential duplicate invoice records
using DuckDB and SQL.

### Business Value
The query helps auditors identify invoice
records requiring further investigation.

### Limitation
Duplicate indicators do not prove fraud.
Manual verification is required.


In [59]:

# AuditGuard - Final SQL Validation

total_invoices = len(df)
flagged_invoices = len(suspicious_invoices)

total_amount = df["amount"].sum()
flagged_amount = suspicious_invoices["amount"].sum()

print("AUDITGUARD - DAY 03 REPORT")
print("---------------------------")
print("Total invoices:", total_invoices)
print("Total invoice amount:", total_amount)
print("Invoices requiring review:", flagged_invoices)
print("Amount requiring review:", flagged_amount)

print(
    "Review percentage:",
        round(flagged_invoices / total_invoices * 100, 2),
            "%"
            )


AUDITGUARD - DAY 03 REPORT
---------------------------
Total invoices: 6
Total invoice amount: 12500
Invoices requiring review: 4
Amount requiring review: 3000
Review percentage: 66.67 %



# Day 03 - SQL Duplicate Invoice Detection

## Objective
Identify potential duplicate invoices using SQL.

## Tools
- Python
- Pandas
- DuckDB
- SQL

## SQL Concepts Learned
- SELECT
- GROUP BY
- HAVING
- CTE
- INNER JOIN

## Results
- Total invoices: 6
- Total invoice amount: 12,500
- Invoices requiring review: 4
- Amount requiring review: 3,000
- Review percentage: 66.67%

## Business Insight
Potential duplicate invoices should be reviewed
before payment approval.

## Limitation
Matching invoice records does not prove fraud.
Human verification is required.
